# 06 — Analyse d'erreurs, décision et recommandations

**Projet** : Diagnostic du mode de défaillance machine avec Keras
**Principe** : un classifieur multi-classes ne décide rien par lui-même. L'argmax dit *quel mode
est le plus probable* ; le métier veut savoir *quelle équipe envoyer*, et toutes les erreurs n'ont
pas le même prix. Ce notebook part du verdict de l'évaluateur de production, descend jusqu'aux
confusions et aux alarmes mal diagnostiquées, puis construit la décision : coût minimal et revue
experte des cas incertains.

Le split de **test** n'est utilisé qu'ici — une seule fois — pour rester une estimation honnête.

## Objectifs pédagogiques

1. Lire le verdict contractuel de l'évaluateur de production, objectif par objectif.
1. Situer le modèle entre la règle actuelle et le plafond atteignable.
1. Lire la matrice de confusion comme une carte des causes physiques d'erreur.
1. Remplacer l'argmax par une décision à coût minimal et chiffrer le gain.
1. Choisir un seuil de revue experte sur la courbe couverture / coût.
1. Reconnaître un plafond structurel et le documenter au lieu de le masquer.

**Objectifs transverses du dépôt**

- Compiler un modèle multi-classes Keras : sortie softmax, SparseCategoricalCrossentropy, callbacks natifs.
- Comparer un réseau de neurones au boosting sur les mêmes données, métriques et références.
- Préserver la calibration de la sortie softmax, dont dépend la décision à coût minimal.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet configure loguru au premier `get_logger()` appelé par `src`. On prend la main ici,
# au niveau WARNING : sans cela, chaque cellule d'entraînement noierait ses tableaux sous
# les lignes INFO de production. Les erreurs réelles restent visibles — c'est l'essentiel.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` ------------------------------------
# Les notebooks travaillent sur un échantillon réduit (6000 lignes) : l'exécution complète
# reste sous la minute, tout en conservant des distributions réalistes.
NB_ROWS = 6000

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_ROWS}",
                "seed=42",
                "log_level=WARNING",
                "++train.epochs=12",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks écrivent leurs artefacts dans `outputs/notebooks` (ignoré par git) afin de ne
# jamais écraser ceux produits par `make train`.
NB_PATHS = ProjectPaths.from_root(PROJECT_ROOT / "outputs" / "notebooks").ensure()

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil cible : 0.55)")
print(f"Cible             : {CONFIG.data.target or 'aucune (apprentissage non supervisé)'}")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Lignes (notebook) : {NB_ROWS}")

In [ ]:
from src.data.generators import SyntheticDataGenerator
from src.data.loaders import RawDataLoader

raw_path = PATHS.data_file(CONFIG.data.dataset_name)
if raw_path.exists():
    # Cas nominal : le dataset a été généré par `make data`, on passe par le loader validant.
    raw = RawDataLoader(PATHS, dataset_name=CONFIG.data.dataset_name).load()
    print(f"Dataset lu depuis {raw_path.relative_to(PROJECT_ROOT)}")
else:
    # Le notebook reste exécutable sur un clone frais : on génère en mémoire.
    raw = SyntheticDataGenerator(n_samples=NB_ROWS, seed=CONFIG.data.seed).generate()
    print("data/raw vide : génération synthétique en mémoire (`make data` la persiste)")

raw = raw.head(NB_ROWS).reset_index(drop=True)
print(f"shape = {raw.shape}")
raw.head()

In [ ]:
from src.data.loaders import DatasetSplitter, feature_target_split
from src.features.build_features import FeatureBuilder, select_feature_columns, split_by_dtype
from src.preprocessing.pipelines import PreprocessingPipeline


def prepare_matrices(frame: pd.DataFrame, config: Any) -> dict[str, Any]:
    """Reproduce what ``TrainPipeline`` does, on the notebook-sized dataset.

    La fonction reprend **exactement** l'enchaînement de production : split → feature
    engineering (appris sur train uniquement) → preprocessing (appris sur train uniquement).
    C'est ce qui rend les chiffres de ce notebook comparables à ceux de `make train`.

    Args:
        frame: Raw dataset.
        config: Validated application configuration.

    Returns:
        Mapping with splits, fitted objects and model-ready matrices.
    """
    target = config.data.target
    drop_columns = list(config.data.drop_columns)

    splitter = DatasetSplitter.from_config(config.model_dump(), seed=config.seed)
    splits = splitter.split(frame, target=target)

    builder = FeatureBuilder.from_config(config.model_dump(), target=target)
    if builder.recipes:
        builder.fit(splits.train)
    enriched = {
        "train": builder.transform(splits.train),
        "val": None if splits.val is None else builder.transform(splits.val),
        "test": builder.transform(splits.test),
    }

    train_frame = enriched["train"]
    feature_columns = select_feature_columns(train_frame, drop_columns=drop_columns, target=target)
    numeric, categorical = split_by_dtype(train_frame, feature_columns)
    explicit = config.preprocessing.model_dump().get("columns") or {}
    numeric = list(explicit.get("numeric") or numeric)
    categorical = list(explicit.get("categorical") or categorical)

    pipeline = PreprocessingPipeline(
        numeric_features=numeric,
        categorical_features=categorical,
        config=config.preprocessing.model_dump(),
        target=target,
    )
    X_train_frame, y_train = feature_target_split(train_frame, target, drop_columns)
    X_train = pipeline.fit_transform(X_train_frame, y_train)

    def project(split: pd.DataFrame | None) -> tuple[pd.DataFrame | None, Any]:
        if split is None:
            return None, None
        _, labels = feature_target_split(split, target, drop_columns)
        return pipeline.transform(split.loc[:, X_train_frame.columns]), labels

    X_val, y_val = project(enriched["val"])
    X_test, y_test = project(enriched["test"])

    # Le pré-traitement supprime la colonne de groupe (identifiant, non modélisable), or une
    # métrique de classement se calcule **par groupe** puis se moyenne. Elle doit donc voyager à
    # côté des matrices, exactement comme dans `TrainPipeline` et dans les fixtures de tests.
    # `None` pour toute tâche sans structure de groupe : le comportement des autres projets est
    # inchangé.
    group_column = getattr(config.data, "group_column", None)

    def groups_of(split: pd.DataFrame | None) -> Any:
        if split is None or not group_column or group_column not in split.columns:
            return None
        return split[group_column].to_numpy()

    return {
        "splits": splits,
        "enriched": enriched,
        "builder": builder,
        "pipeline": pipeline,
        "numeric": numeric,
        "categorical": categorical,
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "groups_train": groups_of(enriched["train"]),
        "groups_val": groups_of(enriched["val"]),
        "groups_test": groups_of(enriched["test"]),
        "feature_names": list(pipeline.feature_names_out),
        # Colonnes de la matrice **avant** pré-traitement (donc avant one-hot). Indispensables dès
        # qu'un notebook ré-applique le pipeline à un nouveau cadre : sélectionner les colonnes de
        # `X_train` (après one-hot) sur un cadre enrichi lève un KeyError sur les modalités.
        "frame_columns": list(X_train_frame.columns),
    }


PREPARED = prepare_matrices(raw, CONFIG)
print("train :", PREPARED["X_train"].shape)
print("val   :", None if PREPARED["X_val"] is None else PREPARED["X_val"].shape)
print("test  :", PREPARED["X_test"].shape)
print(f"features livrées au modèle : {len(PREPARED['feature_names'])}")
PREPARED["X_train"].head()

In [ ]:
from src.models import build_model

MODEL = build_model(CONFIG, feature_names=PREPARED["feature_names"])
# Les groupes (l'identité de l'utilisateur en classement) voyagent à côté des matrices : le
# pré-traitement supprime cette colonne, or une métrique par groupe ne peut pas s'en passer.
FIT_RESULT = MODEL.fit(
    PREPARED["X_train"],
    PREPARED["y_train"],
    X_val=PREPARED["X_val"],
    y_val=PREPARED["y_val"],
    groups=PREPARED.get("groups_train"),
    groups_val=PREPARED.get("groups_val"),
    callbacks=[],
)
print(MODEL.summary())
pd.Series(FIT_RESULT.metrics, name="métrique").to_frame("valeur")

In [ ]:
import json

from src.data.generators import SyntheticDataGenerator

# Le plafond oracle n'est calculable que par le générateur : lui seul connaît les vraies
# probabilités. On rejoue donc la génération configurée (mêmes options, même graine, même volume)
# dans `outputs/notebooks/data/raw` : les références y sont publiées exactement comme en
# production, et l'évaluateur du notebook les lira au même endroit que celui du pipeline.
REFERENCE_GENERATOR = SyntheticDataGenerator.from_config(CONFIG.model_dump(), NB_PATHS)
REFERENCE_GENERATOR.run()
GENERATION = json.loads((NB_PATHS.raw_dir / "generation_metadata.json").read_text(encoding="utf-8"))
same_rows = GENERATION["n_samples"] == len(raw)
print(
    f"références calculées sur {GENERATION['n_samples']} alarmes (identiques à `raw` : {same_rows})"
)

REFERENCE_TABLE = pd.DataFrame(
    {
        "référence": [
            "plancher : classe majoritaire",
            "règle actuelle : code automate",
            "plafond : oracle (vraies probabilités)",
        ],
        "macro-F1": [
            GENERATION["f1_macro_baseline_majority"],
            GENERATION["f1_macro_baseline_alarm_code"],
            GENERATION["f1_macro_ceiling_oracle"],
        ],
        "accuracy": [
            GENERATION["accuracy_baseline_majority"],
            GENERATION["accuracy_baseline_alarm_code"],
            GENERATION["accuracy_ceiling_oracle"],
        ],
    }
)
REFERENCE_TABLE.round(4)

## 1. Le verdict de l'évaluateur de production

In [ ]:
from src.evaluation.evaluator import Evaluator

EVALUATOR = Evaluator.from_config(MODEL, CONFIG.model_dump(), NB_PATHS)
RESULT = EVALUATOR.evaluate(
    PREPARED["X_test"], PREPARED["y_test"], split="test", context=PREPARED["enriched"]["test"]
)
verdict = "conforme" if RESULT.is_compliant else "non conforme"
print(f"verdict : {verdict} ({RESULT.extras['objectives_met']} objectifs atteints)")
RESULT.verdict

**Ce qu'il faut retenir**

- La métrique de décision est `f1_macro` = **celle affichée ci-dessus** ; le verdict la complète par six objectifs métier (règle battue, rappel par mode, pannes acquittées, calibration, coût, latence).
- Un objectif « n/a » n'est pas un succès : il signale une donnée manquante (par exemple le code automate absent du contexte).
- Ce verdict est exactement celui du rapport `artifacts/reports/evaluation_report.md` : même objet, mêmes seuils (`diagnosis.objectives`).

## 2. Le modèle entre la règle actuelle et le plafond

In [ ]:
from src.visualization.plots import MulticlassPlots

PLOTS = MulticlassPlots(NB_PATHS.figures_dir)
display(RESULT.references.round(4))
display(Image(PLOTS.references(RESULT), width=560))

**Ce qu'il faut retenir**

- La référence qui compte est la règle actuelle, pas la classe majoritaire : battre un plancher trivial ne prouve rien au métier.
- Le plafond oracle est calculé sur le jeu complet, le modèle sur le test : un modèle légèrement au-dessus du plafond est dans le bruit d'échantillonnage, pas au-delà du possible.
- Une fois le plafond atteint, plus aucun réglage ne rapporte : la valeur restante est dans la décision (section 4) et dans de nouvelles données (capteurs, historique).

## 3. Qualité par mode et confusions

In [ ]:
display(RESULT.per_class.round(4))
display(Image(PLOTS.confusion_matrix(RESULT), width=560))
RESULT.confusions.head(8)

In [ ]:
# Lecture métier des paires confondues, déclarée dans `diagnosis.confusion_notes`.
notes = {
    frozenset(part.strip() for part in key.split("/")): note
    for key, note in (CONFIG.model_dump()["diagnosis"].get("confusion_notes") or {}).items()
}
for _, row in RESULT.confusions.head(6).iterrows():
    pair = frozenset({row["classe réelle"], row["classe prédite"]})
    reading = notes.get(pair, "pas de lecture documentée")
    print(f"- {row['classe réelle']} -> {row['classe prédite']} ({row['effectif']} alarmes)")
    print(f"  {reading}")

**Ce qu'il faut retenir**

- Une matrice de confusion se lit **ligne par ligne** : chaque ligne dit où partent les alarmes d'un mode réel. Les cellules hors diagonale sont des causes, pas des chiffres.
- Les confusions les plus fréquentes suivent les chevauchements physiques vus au notebook 01 (usure et surcharge, thermique et puissance) : le modèle se trompe là où la physique est ambiguë.
- Une confusion vers `false_alarm` est d'une autre nature qu'une confusion entre deux pannes : la première laisse la machine tourner, la seconde envoie seulement la mauvaise équipe.

## 4. Argmax ou décision à coût minimal

In [ ]:
from src.evaluation.decision import DiagnosisSettings

SETTINGS = DiagnosisSettings.resolve(CONFIG.model_dump(), classes=list(MODEL.classes_))
costs = pd.DataFrame(
    SETTINGS.cost_matrix(RESULT.labels), index=RESULT.labels, columns=RESULT.labels
)
print("Matrice de coûts (EUR, vraie classe en ligne, décision en colonne)")
display(costs)
display(RESULT.decision.round(4))
display(Image(PLOTS.decision_costs(RESULT), width=600))

In [ ]:
# Des alarmes où la décision à coût minimal contredit l'argmax : pourquoi ?
frame = RESULT.predictions
changed = frame[frame["decision"] != frame["y_pred"]]
print(
    f"{len(changed)} alarmes ({len(changed) / len(frame):.1%}) où la décision diffère de l'argmax"
)
columns = [
    "y_true",
    "y_pred",
    "decision",
    "confidence",
    *[f"proba_{label}" for label in RESULT.labels],
]
changed[columns].head(8).round(3)

**Ce qu'il faut retenir**

- La décision à coût minimal change la réponse quand le mode le plus probable est « fausse alarme » mais qu'une panne reste plausible : une probabilité de panne de 40 % suffit à justifier un déplacement, parce qu'acquitter une panne coûte vingt fois plus cher.
- Le prix de cette prudence est visible dans le tableau : quelques déplacements inutiles de plus, beaucoup moins de pannes acquittées, et un coût moyen par alarme nettement inférieur.
- La matrice de coûts est un **contrat métier** (`diagnosis.costs`) : la modifier change la décision sans ré-entraîner le modèle.

## 5. Revue experte : combien d'alarmes confier à un humain ?

In [ ]:
display(RESULT.abstention.round(4))
display(Image(PLOTS.abstention(RESULT), width=600))
display(Image(PLOTS.confidence_histogram(RESULT), width=600))
best = RESULT.abstention.loc[RESULT.abstention["cost_per_alarm"].idxmin()]
print(f"seuil configuré                    : {SETTINGS.review_threshold:.2f}")
print(f"seuil le moins coûteux sur le test : {best['threshold']:.2f}")

**Ce qu'il faut retenir**

- Les diagnostics erronés se concentrent aux faibles confiances : c'est ce qui rend la revue experte rentable — on relit peu d'alarmes pour éviter beaucoup d'erreurs.
- Le seuil se choisit sur le **coût**, pas sur l'exactitude : relire coûte aussi (`diagnosis.costs.expert_review`), et au-delà d'un certain seuil on paie des revues pour des cas faciles.
- Le seuil le moins coûteux du test est une information, pas une décision : il se confirme sur la validation avant de modifier `diagnosis.review_threshold`.

## 6. Les probabilités sont-elles fidèles ?

In [ ]:
display(RESULT.calibration)
display(Image(PLOTS.reliability(RESULT), width=520))
print(f"ECE (erreur de calibration top-label) : {RESULT.ece:.3f}")

**Ce qu'il faut retenir**

- Une courbe sous la diagonale signale un modèle **sur-confiant** : il annonce 90 % et se trompe plus d'une fois sur dix. La décision à coût minimal hériterait de cette erreur.
- Un boosting profond est typiquement sur-confiant ; le modèle livré est volontairement petit pour garder des probabilités fidèles (notebook 04, section 6).
- Si l'ECE dérive en production, le correctif est un recalibrage (isotonic sur une fenêtre récente), pas un ré-entraînement complet.

## 7. Le plafond structurel : `random_failure`

In [ ]:
structural = list(SETTINGS.structural_classes)
frame = RESULT.predictions
for label in structural:
    rows = frame[frame["y_true"] == label]
    if rows.empty:
        continue
    print(f"--- {label} : {len(rows)} alarmes de test")
    oracle_recall = GENERATION["recall_per_class_oracle"].get(label, float("nan"))
    print(f"rappel du modèle   : {float((rows['y_pred'] == label).mean()):.3f}")
    print(f"rappel de l'oracle : {oracle_recall:.3f}")
    display(rows["y_pred"].value_counts(normalize=True).round(3).to_frame("part des prédictions"))
    display(rows["decision"].value_counts(normalize=True).round(3).to_frame("part des décisions"))

**Ce qu'il faut retenir**

- Le rappel de l'oracle sur ce mode est nul : **aucun** modèle ne peut faire mieux, puisque rien d'observable ne l'annonce. Ce n'est pas un défaut de réglage à corriger mais une limite à documenter.
- La décision à coût minimal n'acquitte pourtant pas ces alarmes : elle envoie une équipe sur les cas incertains, ce qui limite la casse même sans diagnostic exact.
- Le levier est ailleurs : maintenance préventive, nouveaux capteurs, ou historique de la machine. Le modèle n'est pas le bon outil pour ce mode.

## 8. Où se concentrent les erreurs ?

In [ ]:
frame = RESULT.predictions
for column in [
    name
    for name in ("product_quality", "alarm_code", "shift", "production_line")
    if name in frame.columns
]:
    table = (
        frame.groupby(column)
        .agg(
            alarmes=("is_error", "size"),
            taux_erreur=("is_error", "mean"),
            confiance=("confidence", "mean"),
        )
        .sort_values("taux_erreur", ascending=False)
    )
    print(f"--- erreurs par `{column}`")
    display(table.round(3))
print("--- erreurs les plus confiantes (elles échappent à la revue experte)")
RESULT.errors.head(10)

**Ce qu'il faut retenir**

- Un segment au taux d'erreur nettement supérieur (une gamme, un code automate) est un candidat à une feature dédiée ou à une revue ciblée.
- Les erreurs confiantes sont les plus dangereuses : elles ne partent pas en revue. Les relire une à une est le meilleur moyen de découvrir une variable manquante.
- Une erreur isolée n'est pas un bug ; un groupe d'erreurs de même nature sur le même segment en est un.

## 9. Rapport exécutable

In [ ]:
from src.evaluation.reports import ReportBuilder

REPORTER = ReportBuilder(NB_PATHS, config=CONFIG.model_dump())
WRITTEN = REPORTER.build(RESULT, model=MODEL)
print(f"{len(WRITTEN)} artefacts écrits dans {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")
for index, recommendation in enumerate(REPORTER.recommendations(RESULT), start=1):
    print(f"{index:2d}. {recommendation}")

## 10. Recommandations concrètes

### issues de l'analyse (calculées ci-dessus)

Elles dépendent des mesures : décision à coût minimal, seuil de revue, recalibrage, mode le moins
bien rappelé, confusion la plus fréquente.

### documentées pour ce cas d'usage

1. Router sur la décision **à coût minimal** plutôt que sur l'argmax : une panne réelle classée fausse alarme coûte bien plus cher qu'une équipe dérangée pour rien.
2. Envoyer en revue experte les alarmes dont la confiance est sous le seuil : quelques pourcents d'alarmes relues suppriment une part importante des erreurs coûteuses.
3. Ajouter les features physiques (écart de température, puissance, effort) comme recettes déclarées : c'est le principal levier de performance, bien avant l'hyperparamétrage.
4. Instrumenter les machines sans capteur de vibration : c'est la seule donnée manquante du parc, et elle discrimine usure et surcharge.
5. Surveiller la dérive de la répartition des modes et de `tool_wear_min` : un changement de politique d'outillage déplace toute la distribution.
6. Ne pas promettre de diagnostic sur `random_failure` : documenter le plafond et traiter ces pannes par la maintenance préventive, pas par le modèle.

### plan d'action proposé

| Priorité | Action | Effet attendu | Comment vérifier |
| --- | --- | --- | --- |
| 1 | Router sur la décision à coût minimal | coût par alarme en baisse, pannes acquittées quasi nulles | §4 rejoué sur le test |
| 2 | Revue experte sous le seuil de confiance | erreurs confiantes seules restantes | courbe d'abstention (§5) |
| 3 | Surveiller l'ECE et la répartition des modes | alerte avant la dérive de la décision | `mlops/model-monitoring` |
| 4 | Instrumenter les machines sans capteur de vibration | meilleure séparation usure / surcharge | notebook 04, ablation |
| 5 | Rejouer ce notebook à chaque nouvelle version de données | non-régression | `make evaluate` + CI |

## 11. Limites assumées

- Les données sont **synthétiques** : la physique est plausible, les niveaux de performance illustrent une méthode.
- Les coûts de `diagnosis.costs` sont indicatifs : seul leur ordre de grandeur relatif est défendable sans données financières réelles.
- `random_failure` n'est pas diagnosticable par construction ; tout gain affiché sur ce mode serait du bruit.
- Une seule passe d'évaluation sur le test : la dispersion est mesurée au notebook 04 (graines), pas ici.